# SAAM Project — Parts III & IV reproducible notebook

This notebook reproduces every result, figure and table associated with the
carbon-aware allocations of the SAAM 2026 project:

* **Part III (Section 3.2 of the PDF)** — long-only minimum-variance portfolio with a 50% lower carbon footprint than the unconstrained minimum-variance portfolio (`mv_50`).
* **Part III (Section 3.3 of the PDF)** — long-only tracking-error portfolio with a 50% lower carbon footprint than the value-weighted benchmark (`vw_50`).
* **Part IV (Section 4 of the PDF)** — long-only tracking-error portfolio following a 10%-per-year carbon-footprint reduction path (`vw_nz`).

The optimisation engine is `src/05_part3_part4.py`. It solves each constrained quadratic program with `scipy.optimize.minimize` (SLSQP) and writes its outputs into `outputs/`. This notebook simply runs that engine and renders the results.

> **Group strategy**: Pacific region, Scope 1 emissions, 50% reduction objective and 10% per-year net-zero path.

## 1. Setup

We resolve the project root regardless of where the notebook is opened from (project root or `src/`), then make `src/` importable.

In [ ]:
from pathlib import Path
import sys
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'src' / '05_part3_part4.py').exists():
    ROOT = Path.cwd().parent
assert (ROOT / 'src' / '05_part3_part4.py').exists(), 'Could not locate project root.'

OUTPUTS = ROOT / 'outputs'
FIGURES = OUTPUTS / 'figures'
OUTPUTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
ROOT

## 2. Run the carbon-aware allocation engine

The cell below imports `src/05_part3_part4.py` as a module and executes its `run()` entry point. The engine:

1. loads cleaned Pacific data (`Clean_Prices_Pacific.csv`, `Clean_Revenues_Pacific.csv`, `Clean_CO2_Scope1_Pacific.csv`) and the raw end-of-year market capitalisation file;
2. for every allocation year $Y \in \{2013, \dots, 2024\}$:
   * builds the eligible universe (≥3 years of monthly returns, ≤50% zero-return ratio, carbon and price valid at end of $Y$);
   * estimates the covariance matrix $\Sigma_Y$ from complete-case rows in the trailing 120-month window with a small ridge for numerical stability;
   * solves the five optimisation problems with SLSQP (closed-form gradients, explicit linear constraints);
   * simulates implementation year $Y+1$ by letting weights drift with monthly realised returns.
3. saves CSV summaries and figures.

We also recompute the value-weighted benchmark with monthly rebalancing (Section 2.3 of the PDF) using the end-of-month capitalisation file. This is the `vw` column in the performance summary; `vw_drift` is the annual-cap, drift-based variant used as benchmark inside the optimisation.

In [ ]:
spec = importlib.util.spec_from_file_location('part3_4', ROOT / 'src' / '05_part3_part4.py')
part3_4 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = part3_4
spec.loader.exec_module(part3_4)
part3_4.run()

## 3. Performance summary

Annualised return, annualised volatility, Sharpe ratio (using a 0% risk-free rate for comparability), worst monthly drawdown, best monthly return and cumulative return over $2014$–$2025$ for every portfolio:

* `vw` — value-weighted, **monthly rebalanced** (PDF Section 2.3, strict reading).
* `vw_drift` — value-weighted at end of each year, drift between rebalancing dates (used as the *benchmark vector* in the tracking-error problems).
* `mv` — long-only minimum-variance.
* `mv_50` — long-only minimum-variance with $CF \le 0.5 \, CF^{(\mathrm{mv})}$.
* `vw_50` — long-only tracking-error portfolio with $CF \le 0.5 \, CF^{(\mathrm{vw})}$.
* `vw_nz` — long-only tracking-error portfolio with $CF \le (1-\theta)^{Y-Y_0+1} CF^{(\mathrm{vw})}_{Y_0}$, $\theta=10\%$.

In [ ]:
performance = pd.read_csv(OUTPUTS / 'part3_4_performance_summary.csv')
performance.set_index('portfolio').round(4)

## 4. Annual carbon metrics

For each year and each portfolio we report the carbon footprint (tCO₂e per USD-million invested), the weighted-average carbon intensity (tCO₂e per USD-million revenue), the carbon cap that applied, and the optimisation status. The carbon footprints of the constrained portfolios should sit at or below the corresponding `carbon_limit`.

In [ ]:
annual = pd.read_csv(OUTPUTS / 'part3_4_annual_carbon_metrics.csv')
annual.head(15)

In [ ]:
# Constraint check: largest excess of the carbon footprint above the cap.
constrained = annual[annual['carbon_limit'].notna() & (annual['portfolio'] != 'vw_drift')].copy()
constrained['excess_over_limit'] = constrained['carbon_footprint_tco2e_per_musd_invested'] - constrained['carbon_limit']
constrained.groupby('portfolio')['excess_over_limit'].agg(['max', 'mean'])

## 5. Top-10 WACI drivers (value-weighted benchmark)

Firms ranked by their WACI contribution `w_vw_i × CI_i` each year. This is the most useful diagnostic for Section 3.1: which firms push the benchmark's WACI up. A high-CI firm with negligible benchmark weight does **not** dominate the WACI.

In [ ]:
drivers = pd.read_csv(OUTPUTS / 'part3_4_top10_waci_drivers_by_year.csv')
drivers.head(20)

## 6. Composition changes — exclusions and overweights vs the VW benchmark

For each constrained portfolio and year we report (i) the firms held by VW but excluded by the constrained portfolio (top 10 by VW weight, i.e. the ones whose exclusion is most consequential), and (ii) the largest overweights vs VW. This is the empirical answer to Section 3.2/3.4 of the PDF ("main changes regarding the composition of the portfolio").

In [ ]:
excl = pd.read_csv(OUTPUTS / 'part3_4_exclusions_overweights.csv')
# Example: exclusions / overweights for the net-zero portfolio in the latest allocation year.
latest_year = excl['year'].max()
excl[(excl['portfolio'] == 'vw_nz') & (excl['year'] == latest_year)].head(20)

## 7. Sector / country tilts induced by the carbon constraints

Country shares aggregated from the per-firm weights. The carbon-constrained portfolios typically tilt away from countries whose investible universe is dominated by high-emission firms.

In [ ]:
weights = pd.read_csv(OUTPUTS / 'part3_4_portfolio_weights.csv')
country_mix = (weights
               .groupby(['year', 'portfolio', 'country'])['weight']
               .sum()
               .unstack('country')
               .fillna(0.0))
country_mix.round(3).tail(15)

## 8. Figures

Cumulative-performance and carbon-evolution figures, ready for the report.

In [ ]:
for filename in [
    'part3_mv_vs_mv50_cumulative.png',
    'part3_vw_vs_vw50_cumulative.png',
    'part4_vw_vw50_netzero_cumulative.png',
    'part3_4_waci_by_portfolio.png',
    'part3_4_carbon_footprint_by_portfolio.png',
    'part4_netzero_path.png',
]:
    display(Markdown(f'### {filename}'))
    display(Image(filename=str(FIGURES / filename)))

## 9. Discussion — financial vs carbon trade-off

Comparing the five portfolios:

* `mv` already produces higher annualised return and Sharpe ratio than `vw`. However, its carbon footprint is structurally **higher** than the benchmark in every allocation year of the sample, because minimum-variance happens to pick stable, capital-intensive, often carbon-heavy firms (utilities, materials).
* `mv_50` achieves exactly the 50% carbon reduction by construction. Because the unconstrained `mv` was very carbon-heavy, halving its footprint still leaves a portfolio with carbon larger than the benchmark in most years. Surprisingly, in our sample `mv_50` has a *higher* Sharpe ratio than `mv`: forcing the optimiser to diversify out of carbon-heavy stocks acts as a side-by-side risk control.
* `vw_50` cuts the benchmark footprint in half while keeping a small tracking error. The cost is modest: a small increase in volatility and a roughly comparable return.
* `vw_nz` follows a tightening path. By 2024 the cap is $(0.9)^{12} \approx 28\%$ of the 2013 benchmark footprint — close to a 70% reduction in a decade. The realised cumulative return remains close to `vw_50`, but the Sharpe ratio is slightly lower because the constraint forces ever larger deviations from the benchmark, raising the tracking error.

The economic intuition is that early in the sample the carbon constraint is loose (the benchmark footprint already shrinks for free as the universe expands and tilts toward services/tech). Later in the sample the constraint binds harder and starts costing return.

## 10. Limitations

* Carbon data coverage is incomplete and forward-filled, which can understate year-on-year changes in real emissions.
* The covariance matrix is estimated on monthly returns, with at most ten years of history; expected results are sensitive to estimation error, especially for new firms with short histories.
* The strategy uses Scope 1 emissions only; switching to Scope 2 or Scope 1+2 changes which firms drive WACI (set `CARBON_SCOPE` in `src/05_part3_part4.py`).
* The carbon constraint can become binding at sector level and create concentrated bets (e.g., financials, healthcare). Sector concentration risk is not penalised explicitly in the optimisation.
* The net-zero path is defined relative to the 2013 benchmark footprint. Results would change with a different base year or a stricter $\theta$.

## Use of Large Language Models (LLMs)

We used Claude (Anthropic) and ChatGPT (OpenAI) as coding-support tools. Their use was limited to: (i) clarifying the algebra of the constraints in the project handout, (ii) debugging Python and pandas indexing issues, (iii) suggesting that a black-box QP solver (`scipy.optimize.minimize` with SLSQP) is more reliable than a hand-rolled projected-gradient routine for this class of problems, and (iv) polishing the language of this notebook. All methodological choices (estimation window, eligibility filters, simulation logic, carbon-footprint formulas, optimisation problems), all coding decisions, and all interpretations are our own. The group is fully responsible for the correctness of the results and for academic integrity.